In [1]:
import sys
!{sys.executable} -m pip install pandas matplotlib openpyxl
%pip install nibabel

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ttest_rel
import os

**SETTINGS**

In [ ]:
excel_files_B = ['/path/to/baseline_model_inference/metrics_original.csv',
                 '/path/to/baseline_model_inference/metrics_level_1.csv',
                 '/path/to/baseline_model_inference/metrics_level_2.csv']


excel_files_DR = ['/path/to/domain_randomisation_model_inference/metrics_original.csv',
                  '/path/to/domain_randomisation_model_inference/metrics_level_1.csv',
                  '/path/to/domain_randomisation_model_inference/metrics_level_2.csv']

files = {
    'level_0': '/path/to/baseline_model_inference/metrics_original.csv', # or '/path/to/domain_randomisation_model_inference/metrics_original.csv'
    'level_1': '/path/to/baseline_model_inference/metrics_level_1.csv',  # or '/path/to/domain_randomisation_model_inference/metrics_level_1.csv'
    'level_2': '/path/to/baseline_model_inference/metrics_level_1.csv'   # or '/path/to/domain_randomisation_model_inference/metrics_level_2.csv'
}
segments = ['whole_tumor', 'Ventral_DC', 'Accumbens_Area', 'Amygdala',  'Hippocampus',  'Brain_Stem ', 'Ventricle_4', 'Ventricle_3', 'Pallidum',  
            'Putamen',  'Caudate',  'Thalamus', 'Cerebellum_Cortex', 'CWM', 'Infe_Lateral_Vent', 'Lateral_Vent', 'Cere_Cor', 'whole_bg', 'enhancing_tumor', 'necrotic_core' , 'inner_bg']

# metrics csv
df = pd.read_csv('/path/to/csv_file_for_analysis') 
modalities = ['T1', 'T1C', 'FLAIR']

**IMAGE QUALITY METRICS PER MODEL**\
Output: \
Image Quality Metrics (MSE, PSNR, SSIM)\
&nbsp;&nbsp;  per modality (T1, T1C, FLAIR),\
&nbsp;&nbsp; grouped by transformation level (level_0, level_1, level_2),\
&nbsp;&nbsp; for baseline or domain randomisation model, based on variable files

In [ ]:
metric_cols = ['mse_3d', 'psnr_3d', 'ssim_3d']
all_stats = {}  # store results per level

for level, filepath in files.items():
    print(f"\n{'='*60}")
    print(f"Processing: {level} → {filepath}")
    print(f"{'='*60}")

    # Load CSV
    df = pd.read_csv(filepath)

    # Convert non-numeric to NaN
    for col in metric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Group by modality and compute mean + std
    stats_df = df.groupby('modality')[metric_cols].agg(['mean', 'std'])

    # Flatten column names
    stats_df.columns = [f'{col}_{stat}' for col, stat in stats_df.columns]

    # Round to 3 decimal places
    stats_df = stats_df.round(3)

    print(stats_df)

    # Store results
    all_stats[level] = stats_df

**SEGMENTATION BASED METRICS**\
Output: \
Segmentation-based metrics per produced segment, computed per modality and transformation level for baseline or domain randomisation model, based on variable files. \
Reported metrics include: \
&nbsp;&nbsp;&nbsp; volume (ml) for ground truth and prediction\
&nbsp;&nbsp;&nbsp; mean and std absolute volume difference \
&nbsp;&nbsp;&nbsp; mean and std signed volume difference\
&nbsp;&nbsp;&nbsp; mean and std absolute relative error (%).

In [ ]:
output_folder = '/path/to/output' 
os.makedirs(output_folder, exist_ok=True)

combined_filename = 'vol_by_modality_<model_name>.csv'

os.makedirs(output_folder, exist_ok=True)


all_summaries = []

for file_label, file_path in files.items():
    df = pd.read_csv(file_path)

    vol_diff_cols = [col for col in df.columns if col.endswith('_vol_diff_ml')]

    summary_parts = []

    for diff_col in vol_diff_cols:
        name     = diff_col[:-len('_vol_diff_ml')]
        gt_col   = f'gt_{name}_vol_ml'
        pred_col = f'pred_{name}_vol_ml'

        df[diff_col] = pd.to_numeric(df[diff_col], errors='coerce')

        agg_dict = {}


        agg_dict[f'{diff_col}_mean'] = (diff_col, 'mean')


        agg_dict[f'{diff_col}_std'] = (diff_col, 'std')

        if gt_col in df.columns:
            df[gt_col] = pd.to_numeric(df[gt_col], errors='coerce')
            agg_dict[f'{gt_col}_mean'] = (gt_col, 'mean')

        if pred_col in df.columns:
            df[pred_col] = pd.to_numeric(df[pred_col], errors='coerce')

            agg_dict[f'{pred_col}_mean'] = (pred_col, 'mean')

        if gt_col in df.columns and pred_col in df.columns:

            signed_diff_col = f'{name}_signed_vol_diff_ml'
            df[signed_diff_col] = df[gt_col] - df[pred_col]
            agg_dict[f'{signed_diff_col}_mean'] = (signed_diff_col, 'mean')
            agg_dict[f'{signed_diff_col}_std']  = (signed_diff_col, 'std')

          
            abs_pct_col = f'{name}_abs_vol_diff_pct'
            df[abs_pct_col] = (df[gt_col] - df[pred_col]).abs() / df[gt_col] * 100
            df[abs_pct_col] = df[abs_pct_col].replace([float('inf'), -float('inf')], float('nan'))  # avoid div by zero
            agg_dict[f'{abs_pct_col}_mean'] = (abs_pct_col, 'mean')
            agg_dict[f'{abs_pct_col}_std']  = (abs_pct_col, 'std')

        part = df.groupby('modality').agg(**agg_dict)
        summary_parts.append(part)

    stats_df = pd.concat(summary_parts, axis=1).reset_index()

    # Round numeric columns to 3 decimals (except pct columns → 2 decimals)
    pct_cols     = [c for c in stats_df.columns if '_pct' in c]
    non_pct_cols = [c for c in stats_df.select_dtypes(include='number').columns if '_pct' not in c]

    stats_df[non_pct_cols] = stats_df[non_pct_cols].round(3)
    stats_df[pct_cols]     = stats_df[pct_cols].round(2)

    # Add file label column
    stats_df.insert(0, 'file', file_label)

    # Save per-file CSV
    out_path = os.path.join(output_folder, f'vol_diff_summary_{file_label}.csv')
    stats_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

    all_summaries.append(stats_df)


# Combine all files into one
combined_df = pd.concat(all_summaries, ignore_index=True)
combined_path = os.path.join(output_folder, combined_filename)
combined_df.to_csv(combined_path, index=False)

print(f"\nCombined file saved: {combined_path}")
print(combined_df)

**VOLUME DIFFERENCE PLOT FOR SELECTED SEGMENTATION**

In [ ]:
all_dfs = []
segment = 'Hippocampus' # Segmentation Name
for level_name, file_name in files.items():
    df = pd.read_csv(file_name)

    gt_col = f'gt_{segment}_vol_ml'
    pred_col = f'pred_{segment}_vol_ml'

    # Convert to numeric in case there are empty cells
    df[gt_col] = pd.to_numeric(df[gt_col], errors='coerce')
    df[pred_col] = pd.to_numeric(df[pred_col], errors='coerce')

    # Calculate difference
    df['vol_diff'] = df[gt_col] - df[pred_col]
    df['level'] = level_name

    all_dfs.append(df)

# Combine all levels into one dataframe
df_all = pd.concat(all_dfs, ignore_index=True)

modalities = ['T1', 'T1C', 'FLAIR']
colors = {
    'level_0': 'steelblue',
    'level_1': 'tomato',
    'level_2': 'seagreen'
}

for modality in modalities:
    subset = df_all[df_all['modality'] == modality]

    fig, ax = plt.subplots(figsize=(7, 5))

    for level_name in files.keys():
        level_subset = subset[subset['level'] == level_name]
        ax.scatter(
            level_subset['fu_time'],
            level_subset['vol_diff'],
            s=60,
            color=colors[level_name],
            label=level_name
        )

    ax.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax.set_title(f'{modality} - {segment} GT vs Pred Difference',
                 fontsize=16, fontweight='bold')
    ax.set_xlabel('Days after treatment', fontsize=13, fontweight='bold')
    ax.set_ylabel('Volume GT - Pred (ml)',  fontsize=13, fontweight='bold')
    ax.tick_params(axis='both', labelsize=11)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(
        title='Level',
        title_fontsize=13,
        fontsize=12,
        prop={'weight': 'bold'}
    )

    plt.tight_layout()
    #Save Optionally
    #plt.savefig(f'{segment}_{modality}_levels_plot.png', dpi=150)
    plt.show()

**DICE SCORE**
Output: \
Dice Score for selected segmentation\
&nbsp;&nbsp;  per modality (T1, T1C, FLAIR),\
&nbsp;&nbsp; grouped by transformation level (level_0, level_1, level_2),\
&nbsp;&nbsp; for baseline or domain randomisation model, based on variable files

In [ ]:
# Select segments to include
selected_segments = ['Hippocampus']  # Change or expand as needed
# selected_segments = segments  # Uncomment to use all

all_dice_stats = {}

for level, filepath in files.items():
    print(f"\n{'='*60}")
    print(f"Processing: {level} → {filepath}")
    print(f"{'='*60}")

    df = pd.read_csv(filepath)

    # Filter to selected dice columns that exist in the file
    selected_dice_cols = [f'dice_{seg}' for seg in selected_segments if f'dice_{seg}' in df.columns]

    if not selected_dice_cols:
        print(f"  No matching dice columns found for selected segments in {level}.")
        continue

    for col in selected_dice_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    stats_df = df.groupby('modality')[selected_dice_cols].agg(['mean', 'std'])
    stats_df.columns = [f'{col}_{stat}' for col, stat in stats_df.columns]

    # Round and reset index
    stats_df = stats_df.round(3).reset_index()
    stats_df['level'] = level  # track which level the stats came from

    print(stats_df)
    all_dice_stats[level] = stats_df

# Optional: combine all levels into one DataFrame
combined = pd.concat(all_dice_stats.values(), ignore_index=True)
print(f"\n{'='*60}")
print("Combined stats across all levels:")
print(combined)

# Optional save
# combined.to_csv('/path/to/output/dice_summary_by_modality_all_levels.csv', index=False)

**JACOBIAN PLOTS**\
Outout: \
Jacobian maps of the ground truth, baseline model prediction, and domain randomisation model prediction for each modality, for a selected session from the test set.

In [ ]:
jacob_paths = { "T1 - Ground Truth" :'/path/to/baseline_model_inference/predictions/original/jacobians/<sub_name>/<ses_name>/anat/<T1_gt_jacobian_filename>.nii.gz',
               'T1 - Baseline Pred': '/path/to/baseline_model_inference/predictions/predictions/original/jacobians/<sub_name>/<ses_name>/<T1_pred_jacobian_filename>.nii.gz',
               "T1 - Domain Rand Pred": '/path/to/domain_randomisation_model_inference/predictions/predictions/original/jacobians/<sub_name>/<ses_name>/anat/<T1_pred_jacobian_filename>.nii.gz',
               'T1C - Ground Truth': '/path/to/baseline_model_inference/predictions/predictions/original/jacobians/<sub_name>/<ses_name>/anat/<T1C_gt_jacobian_filename>.nii.gz',
               'T1C - Baseline Pred': '/path/to/baseline_model_inference/predictions/predictions/original/jacobians/<sub_name>/<ses_name>/anat/<T1C_pred_jacobian_filename>.nii.gz',
               'T1C - Domain Rand Pred': '/path/to/domain_randomisation_model_inference/predictions/predictions/original/jacobians/<sub_name>/<ses_name>/anat/<T1C_pred_jacobian_filename>.nii.gz',
               'FLAIR - Ground Truth': '/path/to/baseline_model_inference/predictions/predictions/original/jacobians/<sub_name>/<ses_name>/anat/<FLAIR_gt_jacobian_filename>.nii.gz',
               'FLAIR - Baseline Pred': '/path/to/baseline_model_inference/predictions/predictions/original/jacobians/<sub_name>/<ses_name>/anat/<FLAIR_pred_jacobian_filename>.nii.gz.nii.gz',
               'FLAIR - Domain Rand Pred': '/path/to/domain_randomisation_model_inference/predictions/predictions/original/jacobians/<sub_name>/<ses_name>/anat/<FLAIR_pred_jacobian_filename>.nii.gz.nii.gz'

         }

show_colorbar = False  # no inline colorbar on each image
save_figures  = False  # set True to save plots
output_folder = "jacobian_plots"

if save_figures:
    os.makedirs(output_folder, exist_ok=True)

def safe_filename(text):
    return "".join(c if c.isalnum() or c in (" ", "_", "-") else "_" for c in text).strip().replace(" ", "_")


for title, path in jacob_paths.items():
    img       = nib.load(path)
    data      = img.get_fdata()
    data_norm = np.clip((data + 1.0) / 2.0, 0, 1)

    slice_idx  = data_norm.shape[2] // 2
    slice_axial = np.rot90(data_norm[:, :, slice_idx])

    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(slice_axial, cmap='jet', vmin=0, vmax=1)

    if show_colorbar:
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Normalized Jacobian', fontsize=18, fontweight='bold')
        cbar.set_ticks([0, 0.5, 1.0])
        cbar.set_ticklabels(['0', '0.5', '1'])
        cbar.ax.tick_params(labelsize=16)

    ax.set_title(title, fontsize=24, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()

    if save_figures:
        fig.savefig(
            os.path.join(output_folder, f"{safe_filename(title)}.png"),
            dpi=300, bbox_inches='tight'
        )

    plt.show()


# Plot colorbar as separate figure
fig_cb, ax_cb = plt.subplots(figsize=(2, 8))  # narrower width
sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])

cbar = plt.colorbar(sm, cax=ax_cb)
#cbar.set_label('Normalized Jacobian', fontsize=24, fontweight='bold', labelpad=18)
cbar.set_ticks([0, 0.5, 1.0])
cbar.set_ticks([0, 0.5, 1.0])

# Make tick labels bigger and bold
cbar.set_ticklabels(['0', '0.5', '1'], fontsize=30, fontweight='bold')

plt.tight_layout()

if save_figures:
    fig_cb.savefig(os.path.join(output_folder, "colorbar.png"), dpi=300, bbox_inches='tight')

plt.show()